In [1]:
# PySpark Imports
import pyspark
from pyspark.sql import SparkSession

# ML Classifier Imports
from pyspark.ml.classification import LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler, StringIndexer, PCA
from pyspark.ml.classification import OneVsRest
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql.functions import mean, col
import time
import os
import sys

In [2]:
# Initialize Spark session
spark = SparkSession.builder.appName("ce53") \
    .master("local[*]") \
    .config("spark.driver.cores", "2") \
    .config("spark.driver.memory", "14g") \
    .config("spark.executor.memory", "14g") \
    .config("spark.executor.cores", "2") \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", "true") \
    .config("spark.dynamicAllocation.enabled", "true") \
    .config("spark.dynamicAllocation.minExecutors", "2") \
    .config("spark.dynamicAllocation.maxExecutors", "2") \
    .config("spark.executor.instances", "2") \
    .config("spark.kryoserializer.buffer.max", "2047m") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.executor.heartbeatInterval","11999s") \
    .config("spark.network.timeout","12000s") \
.getOrCreate()

24/04/15 06:32:26 WARN Utils: Your hostname, colin-MS-7977 resolves to a loopback address: 127.0.1.1; using 192.168.0.164 instead (on interface wlx3c52a1d3ccda)
24/04/15 06:32:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/15 06:32:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
parquet_files = ["Parquet/part-00000-1da06990-329c-4e38-913a-0f0aa39b388d-c000.snappy.parquet", "Parquet/part-00000-26e9208e-7819-451b-b23f-2e47f6d1e834-c000.snappy.parquet", 
                 "Parquet/part-00000-36240b61-b84f-4164-a873-d7973e652780-c000.snappy.parquet", "Parquet/part-00000-3f86626a-1225-47f9-a5a2-0170b737e404-c000.snappy.parquet",
                 "Parquet/part-00000-7c2e9adb-5430-4792-a42b-10ff5bbd46e8-c000.snappy.parquet", "Parquet/part-00000-b1a9fc13-8068-4a5d-91b2-871438709e81-c000.snappy.parquet",
                 "Parquet/part-00000-cbf26680-106d-40e7-8278-60520afdbb0e-c000.snappy.parquet", "Parquet/part-00000-df678a79-4a73-452b-8e72-d624b2732f17-c000.snappy.parquet"]

In [4]:
# Read the parquet files into a dataframe
df = spark.read.parquet(*parquet_files, inferSchema=True)

In [5]:
# Get unique labels and their counts
label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the results
label_counts.show()

+--------------------+-------+
|        label_tactic|  count|
+--------------------+-------+
|   Credential Access|     31|
|     Defense Evasion|      1|
|           Discovery|   2086|
|        Exfiltration|      7|
|      Initial Access|      1|
|    Lateral Movement|      4|
|         Persistence|      1|
|Privilege Escalation|     13|
|      Reconnaissance|9278722|
|Resource Development|      3|
|                none|9281599|
+--------------------+-------+



In [6]:
start_time = time.time()

# List of labels to drop
labels_to_drop = ["Defense Evasion", "Exfiltration", "Initial Access", "Lateral Movement", "Persistence", "Privilege Escalation", "Resource Development", "Credential Access"]

# Filter out the rows with labels to drop
df = df.filter(~col("label_tactic").isin(labels_to_drop))

# Get unique labels and their counts after filtering
filtered_label_counts = df.groupBy("label_tactic").count().orderBy("label_tactic")

# Show the filtered results
filtered_label_counts.show()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

+--------------+-------+
|  label_tactic|  count|
+--------------+-------+
|     Discovery|   2086|
|Reconnaissance|9278722|
|          none|9281599|
+--------------+-------+

Execution time: 1.9908080101013184 seconds


In [7]:
start_time = time.time()



df = df.withColumn("datetime", col("datetime").cast("string"))

# Define columns to index
columns_to_index = ['service', 'conn_state', 'history', 'proto', 'dest_ip_zeek', 'community_id', 'uid', 'src_ip_zeek', 'label_tactic', 'datetime']

# Impute null values with 'null' string
for column in columns_to_index:
    df = df.fillna('null', subset=[column])

# Apply StringIndexer to each column
indexers = [StringIndexer(inputCol=column, outputCol=column+"_indexed").fit(df) for column in columns_to_index]

# Chain indexers together
pipeline = Pipeline(stages=indexers)

# Fit and transform the data
df_indexed = pipeline.fit(df).transform(df)

# Drop original columns
df_indexed = df_indexed.drop(*columns_to_index)

# Show the schema of the DataFrame
#df_indexed.show()



end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 64.71129202842712 seconds


In [8]:
# Split the data into training and test sets
start_time = time.time()

train_data, test_data = df_indexed.randomSplit([0.7, 0.3], seed=42)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.01945781707763672 seconds


In [9]:
from pyspark.ml.feature import Imputer


start_time = time.time()

# List of numeric column names
numeric_columns = ['resp_pkts', 'orig_ip_bytes', 'missed_bytes', 'duration', 'orig_pkts',
                   'resp_ip_bytes', 'dest_port_zeek', 'orig_bytes', 'resp_bytes',
                   'src_port_zeek', 'ts']


# Create an Imputer object
imputer = Imputer(
    inputCols=numeric_columns,
    outputCols=["{}_imputed".format(column) for column in numeric_columns]
)

# Fit the imputer to the training data
imputer_model = imputer.setStrategy("mean").fit(train_data)

# Apply the imputer to the training data
train_data_imputed = imputer_model.transform(train_data)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()


# Apply the imputer to the test data
test_data_imputed = imputer_model.transform(test_data)

# Show updated DataFrames
#train_data_imputed.show()
#test_data_imputed.show()



end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/15 06:33:52 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB


Execution time: 55.77102327346802 seconds
Execution time: 0.02678847312927246 seconds


In [10]:
from pyspark.ml.feature import VectorAssembler


start_time = time.time()

# List of columns to assemble
columns_to_assemble = [column for column in train_data_imputed.columns if column.endswith("_imputed")]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform the training DataFrame
train_data_assembled = assembler.transform(train_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()
# Transform the test DataFrame
test_data_assembled = assembler.transform(test_data_imputed)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


# Select only the features and label columns for both training and test sets
train_data_assembled = train_data_assembled.select("features", "label_tactic_indexed")
test_data_assembled = test_data_assembled.select("features", "label_tactic_indexed")

# Show the schema of the assembled training DataFrame
#train_data_assembled.printSchema()

# Show the schema of the assembled test DataFrame
#test_data_assembled.printSchema()

Execution time: 6.336230278015137 seconds
Execution time: 0.07745838165283203 seconds


In [11]:
from pyspark.ml.feature import StandardScaler

start_time = time.time()

# Standardize data on the training set
scaler = StandardScaler(inputCol="features", outputCol="features_normalized", withMean=True, withStd=True)
scaler_model = scaler.fit(train_data_assembled)
train_data_normalized = scaler_model.transform(train_data_assembled)
train_data_normalized = train_data_normalized.select("features_normalized", "label_tactic_indexed")

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/15 06:34:55 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/15 06:36:02 WARN DAGScheduler: Broadcasting large task binary with size 268.3 MiB


Execution time: 82.671226978302 seconds


In [12]:
start_time = time.time()


# Apply the same transformation to the test set
test_data_normalized = scaler_model.transform(test_data_assembled)
test_data_normalized = test_data_normalized.select("features_normalized", "label_tactic_indexed")


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.029705524444580078 seconds


In [13]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd

start_time = time.time()

# Convert Spark DataFrame to Pandas DataFrame
train_pd = train_data_normalized.toPandas()
test_pd = test_data_normalized.toPandas()

# Extract features and labels from Pandas DataFrames
X_train = train_pd['features_normalized'].values.tolist()
y_train = train_pd['label_tactic_indexed'].values.tolist()
X_test = test_pd['features_normalized'].values.tolist()
y_test = test_pd['label_tactic_indexed'].values.tolist()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Define the number of components for LDA
n_components = 1 

start_time = time.time()

# Perform Linear Discriminant Analysis in scikit-learn with the specified number of components
lda = LinearDiscriminantAnalysis(n_components=n_components)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


start_time = time.time()

# Convert the transformed arrays back to Pandas DataFrames
train_pd_lda = pd.DataFrame(X_train_lda, columns=[f'lda_feature_{i+1}' for i in range(n_components)])
test_pd_lda = pd.DataFrame(X_test_lda, columns=[f'lda_feature_{i+1}' for i in range(n_components)])

# Combine the transformed features with the labels
train_pd_lda['label_tactic_indexed'] = y_train
test_pd_lda['label_tactic_indexed'] = y_test

# Convert Pandas DataFrames back to Spark DataFrames
train_lda = spark.createDataFrame(train_pd_lda)
test_lda = spark.createDataFrame(test_pd_lda)


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

#Show the adjusted data
#train_lda.show()
#test_lda.show()

24/04/15 06:36:30 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB
24/04/15 06:40:45 WARN DAGScheduler: Broadcasting large task binary with size 268.4 MiB


Execution time: 415.42067313194275 seconds
Execution time: 121.00640892982483 seconds
Execution time: 317.7315423488617 seconds


In [14]:
start_time = time.time()

# List of columns to assemble
columns_to_assemble = [f'lda_feature_{i+1}' for i in range(n_components)]

# Create the VectorAssembler
assembler = VectorAssembler(inputCols=columns_to_assemble, outputCol="features")

# Transform train_lda
train_lda_assembled = assembler.transform(train_lda)

# Transform test_lda
test_lda_assembled = assembler.transform(test_lda)

# Select only the assembled features and label column for both datasets
train_lda_assembled = train_lda_assembled.select("features", "label_tactic_indexed")
test_lda_assembled = test_lda_assembled.select("features", "label_tactic_indexed")

# Show the schema of the assembled train_lda DataFrame
train_lda_assembled.printSchema()

# Show the schema of the assembled test_lda DataFrame
test_lda_assembled.printSchema()


end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

root
 |-- features: vector (nullable = true)
 |-- label_tactic_indexed: double (nullable = true)

root
 |-- features: vector (nullable = true)
 |-- label_tactic_indexed: double (nullable = true)

Execution time: 0.05398392677307129 seconds


In [15]:
train_lda_assembled.show()
test_lda_assembled.show()

24/04/15 06:50:21 WARN TaskSetManager: Stage 48 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:50:26 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 48 (TID 151): Attempting to kill Python Worker
24/04/15 06:50:26 WARN TaskSetManager: Stage 49 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+--------------------+
|            features|label_tactic_indexed|
+--------------------+--------------------+
|[-2.414443857622149]|                 0.0|
|[-1.9732088892460...|                 0.0|
|[-1.9428673987786...|                 0.0|
|[-1.9584329485970...|                 0.0|
|[-2.728818130787915]|                 0.0|
|[-2.728809154547062]|                 0.0|
|[-2.727152269355389]|                 0.0|
| [-2.72696649932166]|                 0.0|
| [-2.72696649932166]|                 0.0|
|[-2.7288068483101...|                 0.0|
|[-2.728803916500123]|                 0.0|
|[-2.728803916500123]|                 0.0|
|[-2.7271483153837...|                 0.0|
|[-2.7271443610913...|                 0.0|
|[-2.728924376910042]|                 0.0|
|[-2.728924376910042]|                 0.0|
|[-2.728914303150766]|                 0.0|
|  [-2.7282064568978]|                 0.0|
|[-2.728923727611649]|                 0.0|
| [-2.72892207075737]|          

+--------------------+--------------------+
|            features|label_tactic_indexed|
+--------------------+--------------------+
|[-1.9249206202268...|                 0.0|
|[-2.728818130787915]|                 0.0|
|[-2.728806413777673]|                 0.0|
|[-2.728806413777673]|                 0.0|
|[-2.7288158246003...|                 0.0|
|[-2.7288158246003...|                 0.0|
|[-2.7288068483101...|                 0.0|
|[-2.7271483153837...|                 0.0|
|[-2.7271443610913...|                 0.0|
|[-2.728926033826717]|                 0.0|
|[-2.728926033826717]|                 0.0|
|[-2.728914303150766]|                 0.0|
|[-2.7289135736443...|                 0.0|
|[-2.7289135736443...|                 0.0|
|  [-2.7282064568978]|                 0.0|
|[-2.728923727611649]|                 0.0|
| [-2.72892207075737]|                 0.0|
|[-2.7289112674113...|                 0.0|
|[-2.728788524499074]|                 0.0|
|[-2.7287406613235...|          

24/04/15 06:50:30 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 49 (TID 152): Attempting to kill Python Worker


In [16]:
# Create the SVM model
start_time = time.time()
svm = LinearSVC(labelCol="label_tactic_indexed", featuresCol="features", maxIter=10, regParam=0.0, tol=.00001, fitIntercept=True)

# One Vs. Rest
ovr = OneVsRest(classifier=svm, labelCol='label_tactic_indexed')

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



# Fit the model
start_time = time.time()

svm_model = ovr.fit(train_lda_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

24/04/15 06:50:30 WARN TaskSetManager: Stage 50 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.


Execution time: 0.025795459747314453 seconds


24/04/15 06:50:33 WARN TaskSetManager: Stage 53 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:50:38 WARN TaskSetManager: Stage 54 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:50:43 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/04/15 06:50:43 WARN TaskSetManager: Stage 56 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:50:48 WARN TaskSetManager: Stage 58 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:50:48 WARN TaskSetManager: Stage 60 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:50:48 WARN TaskSetManager: Stage 62 contains a task of very large size (31770 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:50:49 WARN TaskSetManag

Execution time: 70.37938380241394 seconds


In [17]:
# Make predictions
start_time = time.time()

predictions = svm_model.transform(test_lda_assembled)

end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

Execution time: 0.1925036907196045 seconds


In [18]:
# Evaluate the model
# Calculate accuracy
start_time = time.time()
evaluator_accuracy = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="accuracy")
accuracy = evaluator_accuracy.evaluate(predictions)
print("Accuracy:", accuracy)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate precision
start_time = time.time()
evaluator_precision = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedPrecision")
precision = evaluator_precision.evaluate(predictions)
print("Precision:", precision)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate recall
start_time = time.time()
evaluator_recall = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="weightedRecall")
recall = evaluator_recall.evaluate(predictions)
print("Recall:", recall)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")

# Calculate F1-score
start_time = time.time()
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label_tactic_indexed", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)
print("F1-Score:", f1_score)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")



print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-Score:", f1_score)

24/04/15 06:51:40 WARN TaskSetManager: Stage 338 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


Accuracy: 0.9998904594489859
Execution time: 77.49510860443115 seconds


24/04/15 06:52:58 WARN TaskSetManager: Stage 340 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:54:12 WARN TaskSetManager: Stage 342 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


Precision: 0.999781122483
Execution time: 74.35920643806458 seconds


24/04/15 06:55:32 WARN TaskSetManager: Stage 344 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


Recall: 0.9998904594489859
Execution time: 79.63919353485107 seconds


F1-Score: 0.9998357849640462
Execution time: 70.0146951675415 seconds
Accuracy: 0.9998904594489859
Precision: 0.999781122483
Recall: 0.9998904594489859
F1-Score: 0.9998357849640462


In [19]:
from pyspark.sql.functions import expr

start_time = time.time()

# Extract Predictions and True Labels
predictions_and_labels = predictions.select("prediction", "label_tactic_indexed")

# Calculate False Positives
false_positives = predictions_and_labels.filter((predictions_and_labels.prediction == 1) & (predictions_and_labels.label_tactic_indexed == 0)).count()

# Calculate True Negatives
true_negatives = predictions_and_labels.filter((predictions_and_labels.prediction == 0) & (predictions_and_labels.label_tactic_indexed == 0)).count()

# Calculate False Positive Rate (FPR)
fpr = false_positives / (false_positives + true_negatives)

print("False Positive Rate:", fpr)
end_time = time.time()
execution_time = end_time - start_time
print("Execution time:", execution_time, "seconds")


24/04/15 06:56:42 WARN TaskSetManager: Stage 346 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.
24/04/15 06:57:24 WARN TaskSetManager: Stage 349 contains a task of very large size (13605 KiB). The maximum recommended task size is 1000 KiB.


False Positive Rate: 3.5896846820975246e-07
Execution time: 77.89547228813171 seconds


In [20]:
spark.sparkContext.stop()